# Mathematical Engineering - Financial Engineering, FY 2025-2026

# Risk Management - Assignment 1: Hedging a Swaption Portfolio

**Case study:** The IR-derivative desk of Polimi Bank has the following positions opened today (15/02/2008):

- Long swaption payer 1m&10y 5y ATM - Notional €650 Mln
- Vanilla 10y IRS fixed rate receiver - Notional €550 Mln


In [48]:
# import libraries 
import numpy as np
import pandas as pd
import datetime as dt

# Project-specific utilities

from utilities.date_functions import (
    business_date_offset,   # offset a date by years/months/days → next business day
    date_series,            # build a coupon-payment schedule between two dates
)
from utilities.ex0_utilities import bootstrap   # discount-curve bootstrap (Assignment 0)
from utilities.ex1_utilities import (
    swaption_price_calculator,  # Black-76 swaption pricer + delta
    irs_proxy_duration,         # modified-duration proxy for IRS DV01
    swap_par_rate,              # par (or forward) swap rate
    swap_mtm,                   # IRS mark-to-market per unit notional
    SwapType,                   #  PAYER or RECEIVER
)

In [ ]:

# Portfolio Parameters

# Swaption: long payer 10y x 5y ATM, notional €650 Mln 
# The swaption label "1m&10y 5y" means:
#   - the swaption expires in 10 years and 0 months from today
#     (the '1m' in the original text appears to be a label artefact;
#      the notebook sets swaption_maturity_m = 0, consistent with a plain 10y expiry)
#   - the underlying forward-starting swap has a 5-year tenor
swaption_maturity_y   = 10      # expiry: years component
swaption_maturity_m   = 0       # expiry: months component
swaption_tenor_y      = 5       # underlying swap tenor (years)
swaption_fixed_leg_freq = 1     # annual fixed-leg coupons
swaption_type         = SwapType.PAYER
swaption_notional     = 650_000_000   # €650 Mln
sigma_black           = 0.7955  # Black implied vol (given)

# --- IRS: vanilla 10y receiver, notional €550 Mln ---
irs_maturity          = 10      # years
irs_fixed_leg_freq    = 1       # annual fixed-leg coupons
irs_notional          = 550_000_000   # €550 Mln
irs_type              = SwapType.RECEIVER   # we RECEIVE fixed, pay float

In [ ]:
# Bootstrap the discount curve dai dati di mercato (CSV) (Assigment0)

# Valuation date: 15 February 2008
today = dt.datetime(2008, 2, 15)

# Settlement date = today + 2 business days (spot convention)
settlement_date = business_date_offset(today, day_offset=2)

# Caricamento dati di mercato dai CSV

# Deposits
# Columns: label (index), Depos (data scadenza), BID, ASK
depo_raw = pd.read_csv("data/depos.csv", index_col=0, header=0)
depo_raw.columns = ["Depos", "BID", "ASK"]
depo = depo_raw[["BID", "ASK"]].copy()
depo.index = pd.to_datetime(depo_raw["Depos"], dayfirst=True)   # index = date di scadenza
depo /= 100.0   # da percentuale a decimale (es. 4.03 → 0.0403)

# Futures: BID/ASK 
# Columns:  futures dates (index), BID, ASK
# I prezzi futures sono quotati come 100 - tasso → convertiamo in tasso decimale
futures_prices = pd.read_csv("data/futures.csv", index_col=0, header=0)
futures_prices.index = pd.to_datetime(futures_prices.index, dayfirst=True)
futures_prices["BID"] = (100.0 - futures_prices["BID"]) / 100.0
futures_prices["ASK"] = (100.0 - futures_prices["ASK"]) / 100.0

# Futures: date Settle and  Expiry 
# Columns : data future (index), Settle, Expiry
futures_dates = pd.read_csv("data/expiry.csv", index_col=0, header=0)
futures_dates.index  = pd.to_datetime(futures_dates.index,       dayfirst=True)
futures_dates["Settle"] = pd.to_datetime(futures_dates["Settle"], dayfirst=True)
futures_dates["Expiry"] = pd.to_datetime(futures_dates["Expiry"], dayfirst=True)

# join BID/ASK + Settle/Expiry in a single data frame 
futures = futures_prices.join(futures_dates[["Settle", "Expiry"]])

# Swap
# Colonne: label (index), Swaps (data scadenza), BID, ASK
swaps_raw = pd.read_csv("data/swaps.csv", index_col=0, header=0)
swaps_raw.columns = ["Swaps", "BID", "ASK"]
swaps = swaps_raw[["BID", "ASK"]].copy()
swaps.index = pd.to_datetime(swaps_raw["Swaps"], dayfirst=True)  # index = expiry dates
swaps /= 100.0   # da % to decimal 

# ---------------------------------------------------------------------------
# Bootstrap della curva di sconto (nessuno shock per la curva base)
# ---------------------------------------------------------------------------
discount_factors, zero_rates = bootstrap(
    reference_date=today,
    depo=depo,
    futures=futures,
    swaps=swaps,
    shock=0.0,
)

print(f"Today:           {today.date()}")
print(f"Settlement date: {settlement_date.date()}")
print(f"Curve has {len(discount_factors)} nodes (from {discount_factors.index[0].date()} "
      f"to {discount_factors.index[-1].date()})")

Today:           2008-02-15
Settlement date: 2008-02-18
Curve has 62 nodes (from 2008-02-15 to 2058-02-19)


/var/folders/gr/m7hhxzcj2ydgpcz2qrckzbch0000gn/T/ipykernel_71988/2426799146.py:19: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  depo.index = pd.to_datetime(depo_raw["Depos"], dayfirst=True)   # index = date di scadenza
/var/folders/gr/m7hhxzcj2ydgpcz2qrckzbch0000gn/T/ipykernel_71988/2426799146.py:26: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  futures_prices.index = pd.to_datetime(futures_prices.index, dayfirst=True)
/var/folders/gr/m7hhxzcj2ydgpcz2qrckzbch0000gn/T/ipykernel_71988/2426799146.py:33: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  futures_dates.index  = pd.t

## Q1: Mark-to-Market the portfolio at the mid-rate curve


In [46]:
# ===========================================================================
# Q1 – Step 1: Compute the forward swap rate (= swaption strike, since ATM)
# ===========================================================================
# The swaption expires in swaption_maturity_y years from today.
# The underlying swap then runs for swaption_tenor_y more years.

# Swaption expiry date (today + 10y)
swaption_expiry = business_date_offset(
    today,
    year_offset=swaption_maturity_y,
    month_offset=swaption_maturity_m,
)

# Underlying swap maturity (today + 10y + 5y = today + 15y)
underlying_expiry = business_date_offset(
    today,
    year_offset=swaption_maturity_y + swaption_tenor_y,
    month_offset=swaption_maturity_m,
)

# Annual fixed-leg payment schedule of the underlying forward-starting swap
# date_series returns [swaption_expiry, +1y, +2y, +3y, +4y, underlying_expiry]
swaption_underlying_fixed_leg_schedule = date_series(
    swaption_expiry, underlying_expiry, swaption_fixed_leg_freq
)

# Forward swap rate: par rate of the forward-starting 5y swap that begins at swaption_expiry.
# We pass the payment dates (excluding start) and the forward start date.
fwd_swap_rate = swap_par_rate(
    swaption_underlying_fixed_leg_schedule[1:],   # payment dates only (no start date)
    discount_factors,
    fwd_start_date=swaption_underlying_fixed_leg_schedule[0],  # = swaption_expiry
)
print(f"Swaption expiry:   {swaption_expiry.date()}")
print(f"Underlying expiry: {underlying_expiry.date()}")
print(f"Forward swap rate (= ATM strike): {fwd_swap_rate:.4%}")

TypeError: unsupported operand type(s) for -: 'NoneType' and 'float'

In [36]:
# ===========================================================================
# Q1 – Step 2: Mark-to-Market the portfolio
# ===========================================================================

# Since the swaption is ATM, the strike equals the forward swap rate
strike = fwd_swap_rate

# ---- Swaption MtM ----
# Black-76 price and delta (w.r.t. the forward swap rate)
swaption_price, swaption_delta = swaption_price_calculator(
    S0=fwd_swap_rate,
    strike=strike,
    ref_date=today,
    expiry=swaption_expiry,
    underlying_expiry=underlying_expiry,
    sigma_black=sigma_black,
    freq=swaption_fixed_leg_freq,
    discount_factors=discount_factors,
    swaption_type=swaption_type,
    compute_delta=True,
)

# ---- IRS MtM ----
# The 10y IRS was entered at the par rate (ATM) → MtM = 0 at inception.
# We compute the current par rate and MtM to confirm.
irs_expiry = business_date_offset(today, year_offset=irs_maturity)
# Annual payment dates for the IRS fixed leg (today+1y, today+2y, ..., today+10y)
irs_fixed_leg_payment_dates = date_series(today, irs_expiry, irs_fixed_leg_freq)[1:]

# Par rate of the 10y IRS at today's curve
irs_rate = swap_par_rate(irs_fixed_leg_payment_dates, discount_factors)

# MtM of the receiver IRS: receive fixed at irs_rate, pay float
# Since the IRS is ATM (entered at par), MtM = 0
irs_mtm = swap_mtm(
    swap_rate=irs_rate,
    fixed_leg_schedule=irs_fixed_leg_payment_dates,
    discount_factors=discount_factors,
    swap_type=irs_type,    # RECEIVER
)

# ---- Portfolio MtM ----
ptf_mtm = swaption_notional * swaption_price + irs_notional * irs_mtm

print("=" * 60)
print("Q1 – Portfolio Mark-to-Market")
print("=" * 60)
print(f"Swaption price (per unit notional): {swaption_price:.6f}")
print(f"Swaption delta (per unit notional): {swaption_delta:.6f}")
print(f"IRS par rate:                       {irs_rate:.4%}")
print(f"IRS MtM (per unit notional):        {irs_mtm:.6f}")
print(f"Portfolio MtM:                      \u20ac{ptf_mtm:,.2f}")
print()

NameError: name 'fwd_swap_rate' is not defined

## Q2: Evaluate the portfolio DV01-parallel


In [37]:
# ===========================================================================
# Q2 – Portfolio DV01-parallel (numerical bump-and-reprice)
# ===========================================================================
# The DV01-parallel is the change in portfolio MtM when ALL market rates are
# shifted up by +1 basis point (1e-4).
#
# Procedure:
#   1. Re-bootstrap the curve with shock = +1bp
#   2. Re-price the swaption and IRS on the shocked curve
#   3. DV01 = shocked_MtM - base_MtM

SHOCK_1BP = 1e-4   # +1 basis point

# Shocked discount curve (parallel +1bp shift to all input rates)
discount_factors_up, _ = bootstrap(
    reference_date=today,
    depo=depo,
    futures=futures,
    swaps=swaps,
    shock=SHOCK_1BP,
)

# Forward swap rate on the shocked curve
fwd_swap_rate_up = swap_par_rate(
    swaption_underlying_fixed_leg_schedule[1:],
    discount_factors_up,
    fwd_start_date=swaption_underlying_fixed_leg_schedule[0],
)

# Swaption price on the shocked curve
# Note: we use the SHOCKED forward rate as S0 (the swaption MtM depends on
# both the forward rate and the annuity, both of which change with the curve)
swaption_price_up = swaption_price_calculator(
    S0=fwd_swap_rate_up,
    strike=strike,                      # strike stays fixed (contractual)
    ref_date=today,
    expiry=swaption_expiry,
    underlying_expiry=underlying_expiry,
    sigma_black=sigma_black,            # vol stays fixed
    freq=swaption_fixed_leg_freq,
    discount_factors=discount_factors_up,
    swaption_type=swaption_type,
    compute_delta=False,
)

# IRS MtM on the shocked curve (the contractual rate irs_rate stays fixed)
irs_mtm_up = swap_mtm(
    swap_rate=irs_rate,
    fixed_leg_schedule=irs_fixed_leg_payment_dates,
    discount_factors=discount_factors_up,
    swap_type=irs_type,
)

# Shocked portfolio MtM
ptf_mtm_up = swaption_notional * swaption_price_up + irs_notional * irs_mtm_up

# DV01-parallel = change in MtM for +1bp parallel shift
ptf_numeric_dv01 = ptf_mtm_up - ptf_mtm

print("=" * 60)
print("Q2 – Portfolio DV01-parallel (numerical)")
print("=" * 60)
print(f"Fwd swap rate (base):   {fwd_swap_rate:.4%}")
print(f"Fwd swap rate (+1bp):   {fwd_swap_rate_up:.4%}")
print(f"Swaption price (base):  {swaption_price:.6f}")
print(f"Swaption price (+1bp):  {swaption_price_up:.6f}")
print(f"IRS MtM (base):         {irs_mtm:.6f}")
print(f"IRS MtM (+1bp):         {irs_mtm_up:.6f}")
print(f"Portfolio DV01-parallel: \u20ac{ptf_numeric_dv01:,.2f}")
print()

TypeError: unsupported operand type(s) for -: 'NoneType' and 'float'

## Q3: Analytical portfolio DV01 approximation


In [38]:
# ===========================================================================
# Q3 – Analytical DV01 approximation
# ===========================================================================
# The proxy DV01 uses:
#   (a) Swaption: dV/dr ≈ N_sw * swaption_delta  (per unit notional, per unit rate change)
#   (b) IRS:      dV/dr ≈ N_irs * irs_duration    (modified-duration proxy)
#
# Combined: ptf_proxy_dv01 = (N_sw * delta_sw + N_irs * D_irs) * 1e-4
#
# The irs_proxy_duration function returns the negative modified duration of
# the equivalent fixed-rate bond (negative because a receiver IRS gains when
# rates fall: dMtM/dr < 0).
#
# For the swaption payer, swaption_delta = BPV*(N(d1)-1) < 0, meaning the
# long payer swaption LOSES value when rates fall (it has positive DV01 – it
# gains when rates rise). Combined with the receiver IRS (which gains when
# rates fall), the portfolio is partially offsetting.

irs_duration = irs_proxy_duration(
    ref_date=today,
    swap_rate=irs_rate,
    fixed_leg_payment_dates=irs_fixed_leg_payment_dates,
    discount_factors=discount_factors,
)

ptf_proxy_dv01 = (
    swaption_notional * swaption_delta
    + irs_notional * irs_duration
) * 1e-4

print("=" * 60)
print("Q3 – Analytical DV01 approximation")
print("=" * 60)
print(f"Swaption delta (per unit notional): {swaption_delta:.6f}")
print(f"IRS proxy duration:                 {irs_duration:.6f}")
print(f"Portfolio proxy DV01:    \u20ac{ptf_proxy_dv01:,.2f}")
print(f"Portfolio numeric DV01:  \u20ac{ptf_numeric_dv01:,.2f}")
print(f"Difference:              \u20ac{ptf_proxy_dv01 - ptf_numeric_dv01:,.2f}")
print()
print("Discussion:")
print("  The analytical (proxy) DV01 approximates each instrument's rate sensitivity")
print("  using first-order derivatives (delta for the swaption, modified duration for")
print("  the IRS). The numerical DV01 captures both first- and second-order (convexity)")
print("  effects via full repricing. For the long payer swaption the proxy underestimates")
print("  the sensitivity because it ignores the positive gamma (convexity). For the IRS,")
print("  the modified-duration proxy is a good approximation for small shocks.")
print()

NameError: name 'irs_rate' is not defined

## Q4: Delta-hedge the portfolio with a 10y IRS


In [39]:
# ===========================================================================
# Q4 – Delta-hedge the portfolio with a 10y IRS
# ===========================================================================
# To make the hedged portfolio DV01 ≈ 0, we add a hedge IRS with notional N_h
# such that:
#   DV01_ptf + N_h * DV01_per_unit_10y_IRS = 0
#
# Numerical approach: find N_h so that:
#   swaption_notional * swaption_price_up
#   + (irs_notional + N_h) * irs_mtm_up
#   + N_h * (irs_mtm_up - 0)   <- hedge IRS is entered ATM → base MtM = 0
#   ≈ ptf_mtm
#
# Equivalently, the total notional of the 10y receiver IRS position (original +
# hedge) must satisfy:
#   [swaption DV01] + [total IRS DV01] = 0
#
# irs_mtm is the MtM change per unit notional for a 1bp shock on the 10y IRS.
# So we need:
#   N_total_irs = -[swaption_DV01_abs] / irs_mtm_up_per_unit
#
# Here irs_mtm_up (= irs_mtm under +1bp) already encodes the DV01 per unit.
# The hedge notional ON TOP of the existing 550 Mln IRS is:
#   N_h = N_total_irs - irs_notional

min_lot = 1_000_000  # IRS traded in multiples of €1 Mln

# DV01 per unit notional of the 10y IRS (receiver)
irs_dv01_per_unit = irs_mtm_up - irs_mtm   # irs_mtm_up uses discount_factors_up

# DV01 of the swaption leg alone
swaption_dv01 = swaption_notional * (swaption_price_up - swaption_price)

# Required TOTAL receiver IRS notional (to zero out the total DV01)
# ptf_dv01 = swaption_dv01 + irs_notional * irs_dv01_per_unit
# 0 = swaption_dv01 + N_total * irs_dv01_per_unit
# => N_total = -swaption_dv01 / irs_dv01_per_unit
N_total_irs_num = -swaption_dv01 / irs_dv01_per_unit

# Hedge notional = total needed - already held
# Round to the nearest €1 Mln lot
N_hedge_raw = N_total_irs_num - irs_notional
N_hedge_num = round(N_hedge_raw / min_lot) * min_lot

# Total IRS notional after the hedge
delta_hedge_swap_notional = irs_notional + N_hedge_num

# Verify: residual DV01 of the hedged portfolio
delta_hedge_dv01 = (
    swaption_notional * swaption_price_up
    + delta_hedge_swap_notional * irs_mtm_up
) - (
    swaption_notional * swaption_price
    + delta_hedge_swap_notional * irs_mtm
)

# --- Analytical hedge (proxy DV01) ---
# Using the proxy: DV01_swaption ≈ N_sw * delta_sw * 1e-4
#                  DV01_IRS      ≈ N_irs * D_irs * 1e-4
# Set total proxy DV01 = 0:
#   N_sw * delta_sw + N_total_irs * D_irs = 0
#   => N_total_irs = -N_sw * delta_sw / D_irs
N_total_irs_proxy = -swaption_notional * swaption_delta / irs_duration
N_hedge_proxy = round((N_total_irs_proxy - irs_notional) / min_lot) * min_lot
delta_hedge_swap_notional_proxy = irs_notional + N_hedge_proxy

print("=" * 60)
print("Q4 – Delta-hedge with a 10y IRS")
print("=" * 60)
print(f"IRS DV01 per unit notional:        \u20ac{irs_dv01_per_unit:.8f}")
print(f"Swaption DV01:                     \u20ac{swaption_dv01:,.2f}")
print()
print(f"Numerical hedge: \u20ac{delta_hedge_swap_notional:,.0f} total IRS notional")
print(f"  (hedge leg: \u20ac{N_hedge_num:,.0f})")
print(f"  Residual DV01: \u20ac{delta_hedge_dv01:,.0f}")
print()
print(f"Analytical hedge: \u20ac{delta_hedge_swap_notional_proxy:,.0f} total IRS notional")
print(f"  (hedge leg: \u20ac{N_hedge_proxy:,.0f})")
print()
print("Discussion:")
print("  The analytical approximation may differ from the numerical hedge because:")
print("  1) The proxy swaption delta ignores convexity (gamma).")
print("  2) The IRS modified-duration proxy is an approximation.")
print("  For a large portfolio the difference can be material.")
print()

NameError: name 'irs_mtm_up' is not defined

## Q5: Portfolio coarse-grained bucket DV01 (10y and 15y buckets)


In [40]:
# ===========================================================================
# Q5 – Coarse-grained bucket DV01
# ===========================================================================
# A bucket DV01 measures the portfolio sensitivity to a +1bp shift in ONE
# segment of the curve, keeping all other maturities unchanged.
#
# Coarse-grained buckets:
#   - 10y bucket: shock ONLY the 10y swap rate (and re-bootstrap)
#   - 15y bucket: shock ONLY the 15y swap rate (and re-bootstrap)
#
# Implementation: build a shock vector (pd.Series indexed by market instrument
# maturity) that is +1bp only for the target tenor and 0 elsewhere.

def make_bucket_shock(swap_index, target_tenor_date, shock_size=1e-4):
    """
    Build a shock Series with +shock_size only at the target swap maturity.
    The index must match the swaps DataFrame index.
    """
    shock = pd.Series(0.0, index=swap_index)
    # Find the swap whose maturity is closest to the target
    closest = min(swap_index, key=lambda d: abs((d - target_tenor_date).days))
    shock[closest] = shock_size
    return shock

# --- 10y bucket shock ---
# The 10y swap matures approximately at today + 10y
date_10y = business_date_offset(settlement_date, year_offset=10)
shock_10y_series = make_bucket_shock(swaps.index, date_10y)

# Combine depo/futures/swap shocks (depos and futures = 0, swaps = bucket)
# The bootstrap function accepts a float (parallel) or a Series aligned to market data.
# We call bootstrap with a shock aligned to swaps only; depos/futures see 0.
discount_factors_10y_up, _ = bootstrap(
    reference_date=today,
    depo=depo,
    futures=futures,
    swaps=swaps,
    shock=shock_10y_series,
)

# --- 15y bucket shock ---
date_15y = business_date_offset(settlement_date, year_offset=15)
shock_15y_series = make_bucket_shock(swaps.index, date_15y)

discount_factors_15y_up, _ = bootstrap(
    reference_date=today,
    depo=depo,
    futures=futures,
    swaps=swaps,
    shock=shock_15y_series,
)

print("Shocked curves built for 10y and 15y coarse-grained buckets.")

KeyError: "None of [DatetimeIndex(['2008-02-20', '2008-02-26', '2008-03-19', '2008-04-21',\n               '2008-05-19'],\n              dtype='datetime64[us]', name='Swaps', freq=None)] are in the [index]"

In [ ]:
# ===========================================================================
# Q5 – Portfolio Coarse-Grained 10y Bucket DV01
# ===========================================================================
# The 10y bucket affects:
#   - The IRS (pays / receives until 10y)
#   - The swaption only through the annuity discounting (the forward rate is
#     determined by the 10y–15y segment of the curve, but the annuity uses
#     the 10y–15y discount factors)
#
# For the swaption we use fwd_swap_rate (unchanged: only 10y shock does NOT
# affect the 10y–15y forward rate) as S0, but re-discount the annuity.

# Forward swap rate under the 10y bucket shock
fwd_swap_rate_10y_up = swap_par_rate(
    swaption_underlying_fixed_leg_schedule[1:],
    discount_factors_10y_up,
    fwd_start_date=swaption_underlying_fixed_leg_schedule[0],
)

swaption_price_10y_up = swaption_price_calculator(
    S0=fwd_swap_rate_10y_up,
    strike=strike,
    ref_date=today,
    expiry=swaption_expiry,
    underlying_expiry=underlying_expiry,
    sigma_black=sigma_black,
    freq=swaption_fixed_leg_freq,
    discount_factors=discount_factors_10y_up,
    swaption_type=swaption_type,
    compute_delta=False,
)

# IRS MtM under 10y bucket shock
irs_mtm_10y_up = swap_mtm(
    swap_rate=irs_rate,
    fixed_leg_schedule=irs_fixed_leg_payment_dates,
    discount_factors=discount_factors_10y_up,
    swap_type=irs_type,
)

ptf_mtm_10y_up = (
    swaption_notional * swaption_price_10y_up
    + irs_notional * irs_mtm_10y_up
)

ptf_numeric_10y_dv01 = ptf_mtm_10y_up - ptf_mtm
print(f"Portfolio Coarse-Grained 10y Bucket DV01: \u20ac{ptf_numeric_10y_dv01:,.2f}")

In [ ]:
# ===========================================================================
# Q5 – Portfolio Coarse-Grained 15y Bucket DV01
# ===========================================================================
# The 15y bucket mainly affects the forward-starting 5y swap (10y→15y) and
# hence the swaption. The 10y IRS is NOT affected by the 15y segment.

# Forward swap rate under the 15y bucket shock
fwd_swap_rate_15y_up = swap_par_rate(
    swaption_underlying_fixed_leg_schedule[1:],
    discount_factors_15y_up,
    fwd_start_date=swaption_underlying_fixed_leg_schedule[0],
)

swaption_price_15y_up = swaption_price_calculator(
    S0=fwd_swap_rate_15y_up,
    strike=strike,
    ref_date=today,
    expiry=swaption_expiry,
    underlying_expiry=underlying_expiry,
    sigma_black=sigma_black,
    freq=swaption_fixed_leg_freq,
    discount_factors=discount_factors_15y_up,
    swaption_type=swaption_type,
    compute_delta=False,
)

# IRS MtM under 15y bucket shock (barely changes; 10y IRS not exposed to 15y)
irs_mtm_15y_up = swap_mtm(
    swap_rate=irs_rate,
    fixed_leg_schedule=irs_fixed_leg_payment_dates,
    discount_factors=discount_factors_15y_up,
    swap_type=irs_type,
)

ptf_mtm_15y_up = (
    swaption_notional * swaption_price_15y_up
    + irs_notional * irs_mtm_15y_up
)

ptf_numeric_15y_dv01 = ptf_mtm_15y_up - ptf_mtm

print("=" * 60)
print("Q5 – Coarse-Grained Bucket DV01")
print("=" * 60)
print(f"Portfolio Coarse-Grained 10y Bucket DV01: \u20ac{ptf_numeric_10y_dv01:,.2f}")
print(f"Portfolio Coarse-Grained 15y Bucket DV01: \u20ac{ptf_numeric_15y_dv01:,.2f}")
print(f"Sum of bucket DV01s:                      \u20ac{ptf_numeric_10y_dv01 + ptf_numeric_15y_dv01:,.2f}")
print(f"Parallel DV01 (Q2):                       \u20ac{ptf_numeric_dv01:,.2f}")
print()
print("Note: the sum of bucket DV01s approximates the parallel DV01; small")
print("differences arise because the coarse-grained bucketing does not perfectly")
print("decompose a parallel shock into two independent bucket shocks.")
print()

## Q6: Delta-hedge with two liquid IRS (10y and 15y)


In [ ]:
# ===========================================================================
# Q6 – Delta-hedge with two IRS: 10y and 15y
# ===========================================================================
# Choice of hedging instruments:
#   - 10y IRS: hedges the 10y bucket risk of the portfolio
#              (mainly from the 10y IRS already in the book)
#   - 15y IRS: hedges the 15y bucket risk
#              (mainly from the swaption whose underlying runs 10y→15y)
#
# These are the most liquid tenors in the EUR swap market and they bracket
# the portfolio's main exposures. The 10y IRS and the 15y IRS are directly
# used in the bootstrap, ensuring no interpolation noise.
#
# System of equations: zero BOTH bucket DV01s
#   DV01_10y_total = 0:   ptf_10y_dv01 + N_h10 * dv01_10y_irs + N_h15 * dv01_15y_of_10y_irs = 0
#   DV01_15y_total = 0:   ptf_15y_dv01 + N_h10 * dv01_10y_of_15y_irs + N_h15 * dv01_15y_irs = 0
#
# In matrix form:  A * [N_h10, N_h15]' = -[ptf_10y_dv01, ptf_15y_dv01]'

# ---- DV01 of a 10y ATM IRS per unit notional in each bucket ----
# We create a new 10y IRS and a new 15y IRS (entered ATM today, base MtM = 0)
irs15_expiry = business_date_offset(today, year_offset=15)
irs15_fixed_leg_dates = date_series(today, irs15_expiry, 1)[1:]
irs15_rate = swap_par_rate(irs15_fixed_leg_dates, discount_factors)

# 10y IRS
# Under 10y bucket shock
irs10_mtm_10y_up = swap_mtm(irs_rate, irs_fixed_leg_payment_dates,
                             discount_factors_10y_up, swap_type=SwapType.RECEIVER)
irs10_mtm_base = swap_mtm(irs_rate, irs_fixed_leg_payment_dates,
                           discount_factors, swap_type=SwapType.RECEIVER)
# Under 15y bucket shock
irs10_mtm_15y_up = swap_mtm(irs_rate, irs_fixed_leg_payment_dates,
                             discount_factors_15y_up, swap_type=SwapType.RECEIVER)

dv01_10y_irs_in_10y_bucket = irs10_mtm_10y_up - irs10_mtm_base
dv01_10y_irs_in_15y_bucket = irs10_mtm_15y_up - irs10_mtm_base

# 15y IRS
# Under 10y bucket shock
irs15_mtm_10y_up = swap_mtm(irs15_rate, irs15_fixed_leg_dates,
                             discount_factors_10y_up, swap_type=SwapType.RECEIVER)
irs15_mtm_base = swap_mtm(irs15_rate, irs15_fixed_leg_dates,
                           discount_factors, swap_type=SwapType.RECEIVER)
# Under 15y bucket shock
irs15_mtm_15y_up = swap_mtm(irs15_rate, irs15_fixed_leg_dates,
                             discount_factors_15y_up, swap_type=SwapType.RECEIVER)

dv01_15y_irs_in_10y_bucket = irs15_mtm_10y_up - irs15_mtm_base
dv01_15y_irs_in_15y_bucket = irs15_mtm_15y_up - irs15_mtm_base

# ---- Portfolio bucket DV01s (ORIGINAL portfolio: swaption + 550M receiver IRS) ----
ptf_10y_dv01_original = ptf_numeric_10y_dv01
ptf_15y_dv01_original = ptf_numeric_15y_dv01

# ---- Solve the 2x2 system ----
# A = [[dv01_10y_irs per unit in 10y bucket, dv01_15y_irs per unit in 10y bucket],
#      [dv01_10y_irs per unit in 15y bucket, dv01_15y_irs per unit in 15y bucket]]
A = np.array([
    [dv01_10y_irs_in_10y_bucket, dv01_15y_irs_in_10y_bucket],
    [dv01_10y_irs_in_15y_bucket, dv01_15y_irs_in_15y_bucket],
])
b = np.array([-ptf_10y_dv01_original, -ptf_15y_dv01_original])

# Hedge notionals
hedge_notionals = np.linalg.solve(A, b)
N_h10_raw, N_h15_raw = hedge_notionals

# Round to nearest €1 Mln
N_h10 = round(N_h10_raw / min_lot) * min_lot
N_h15 = round(N_h15_raw / min_lot) * min_lot

print("=" * 60)
print("Q6 – Delta-hedge with two IRS (10y and 15y)")
print("=" * 60)
print(f"10y IRS DV01 in 10y bucket (per unit): \u20ac{dv01_10y_irs_in_10y_bucket:.8f}")
print(f"10y IRS DV01 in 15y bucket (per unit): \u20ac{dv01_10y_irs_in_15y_bucket:.8f}")
print(f"15y IRS DV01 in 10y bucket (per unit): \u20ac{dv01_15y_irs_in_10y_bucket:.8f}")
print(f"15y IRS DV01 in 15y bucket (per unit): \u20ac{dv01_15y_irs_in_15y_bucket:.8f}")
print()
print(f"Portfolio 10y bucket DV01 (original): \u20ac{ptf_10y_dv01_original:,.2f}")
print(f"Portfolio 15y bucket DV01 (original): \u20ac{ptf_15y_dv01_original:,.2f}")
print()
print(f"Hedge 10y IRS notional: \u20ac{N_h10:,.0f}")
print(f"Hedge 15y IRS notional: \u20ac{N_h15:,.0f}")
print()

# Verify: residual bucket DV01s of the hedged portfolio
residual_10y = (ptf_10y_dv01_original
                + N_h10 * dv01_10y_irs_in_10y_bucket
                + N_h15 * dv01_15y_irs_in_10y_bucket)
residual_15y = (ptf_15y_dv01_original
                + N_h10 * dv01_10y_irs_in_15y_bucket
                + N_h15 * dv01_15y_irs_in_15y_bucket)
print(f"Residual 10y bucket DV01 after hedge: \u20ac{residual_10y:,.2f}")
print(f"Residual 15y bucket DV01 after hedge: \u20ac{residual_15y:,.2f}")
print()

## Q7: Curve flattening scenario


In [ ]:
# ===========================================================================
# Q7 – Curve flattening scenario
# ===========================================================================
# Scenario: 10y rate +1bp, 15y rate -1bp  (curve flattening)
#
# We compute P&L for:
#   (a) Portfolio hedged with a single 10y IRS (Q4)
#   (b) Portfolio hedged with 10y + 15y IRS (Q6)
#
# P&L = MtM_scenario - MtM_base

SHOCK_FLATTEN_10Y = +1e-4   # +1bp on 10y
SHOCK_FLATTEN_15Y = -1e-4   # -1bp on 15y

# Build the shock Series for the flattening scenario:
# +1bp on the 10y swap, -1bp on the 15y swap, 0 elsewhere
shock_flatten = pd.Series(0.0, index=swaps.index)
# Identify 10y and 15y nodes
node_10y = min(swaps.index, key=lambda d: abs((d - date_10y).days))
node_15y = min(swaps.index, key=lambda d: abs((d - date_15y).days))
shock_flatten[node_10y] = SHOCK_FLATTEN_10Y
shock_flatten[node_15y] = SHOCK_FLATTEN_15Y

discount_factors_flatten, _ = bootstrap(
    reference_date=today,
    depo=depo,
    futures=futures,
    swaps=swaps,
    shock=shock_flatten,
)

# ---- Re-price all instruments under the flattening scenario ----
fwd_swap_rate_flat = swap_par_rate(
    swaption_underlying_fixed_leg_schedule[1:],
    discount_factors_flatten,
    fwd_start_date=swaption_underlying_fixed_leg_schedule[0],
)
swaption_price_flat = swaption_price_calculator(
    S0=fwd_swap_rate_flat,
    strike=strike,
    ref_date=today,
    expiry=swaption_expiry,
    underlying_expiry=underlying_expiry,
    sigma_black=sigma_black,
    freq=swaption_fixed_leg_freq,
    discount_factors=discount_factors_flatten,
    swaption_type=swaption_type,
)
irs_mtm_flat = swap_mtm(irs_rate, irs_fixed_leg_payment_dates,
                         discount_factors_flatten, swap_type=irs_type)
irs10_mtm_flat = swap_mtm(irs_rate, irs_fixed_leg_payment_dates,
                           discount_factors_flatten, swap_type=SwapType.RECEIVER)
irs15_mtm_flat = swap_mtm(irs15_rate, irs15_fixed_leg_dates,
                           discount_factors_flatten, swap_type=SwapType.RECEIVER)

# ---- (a) Portfolio hedged with single 10y IRS (Q4) ----
# Original portfolio + hedge 10y IRS of notional N_hedge_num
# The hedge IRS is entered ATM (base MtM = 0), so P&L on hedge = N_hedge * irs10_mtm_flat
pnl_ptf_flat_q4 = (
    swaption_notional * (swaption_price_flat - swaption_price)  # swaption P&L
    + irs_notional * (irs_mtm_flat - irs_mtm)                   # original IRS P&L
    + N_hedge_num * (irs10_mtm_flat - irs10_mtm_base)           # hedge IRS P&L
)

# ---- (b) Portfolio hedged with 10y + 15y IRS (Q6) ----
pnl_ptf_flat_q6 = (
    swaption_notional * (swaption_price_flat - swaption_price)  # swaption P&L
    + irs_notional * (irs_mtm_flat - irs_mtm)                   # original IRS P&L
    + N_h10 * (irs10_mtm_flat - irs10_mtm_base)                 # 10y hedge P&L
    + N_h15 * (irs15_mtm_flat - irs15_mtm_base)                 # 15y hedge P&L
)

print("=" * 60)
print("Q7 – Curve Flattening Scenario (+1bp 10y, -1bp 15y)")
print("=" * 60)
print(f"Fwd swap rate (base):     {fwd_swap_rate:.4%}")
print(f"Fwd swap rate (flatten):  {fwd_swap_rate_flat:.4%}")
print()
print(f"P&L – hedge with single 10y IRS (Q4): \u20ac{pnl_ptf_flat_q4:,.2f}")
print(f"P&L – hedge with 10y + 15y IRS  (Q6): \u20ac{pnl_ptf_flat_q6:,.2f}")
print()
print("Discussion:")
print("""  In a flattening scenario the 15y rate falls while the 10y rate rises.

  (a) Single 10y IRS hedge (Q4):
      The hedge zeros the PARALLEL DV01, but does NOT protect against
      non-parallel (flattening) moves. The swaption (10y→15y underlying)
      is mainly exposed to the 15y bucket: a -1bp in the 15y rate benefits
      the long payer swaption (lower forward rate → lower swaption value,
      but the annuity expands). The 10y receiver IRS gains from the +1bp
      (higher rates → MtM loss for receiver). The residual P&L is non-zero
      because bucket exposures are unhedged.

  (b) Two-IRS hedge (Q6):
      By independently zeroing both the 10y and 15y bucket DV01s, this
      strategy greatly reduces the residual P&L under the flattening
      scenario. A near-zero P&L confirms that bucket-level hedging is
      superior for non-parallel curve moves.

  Further accuracy: using fine-grained bucket DV01s (e.g. 1y, 2y, ..., 30y)
  and hedging each bucket separately would give an even more robust hedge.
  The trade-off is execution cost (more hedging instruments, bid-ask spread)
  and operational complexity vs. the residual risk from coarse bucketing.
""")